# Notebook 1: Premier League match prediction

Two branches that merge into one answer.

**Branch 1** is a neural network trained on twenty seasons of results. It knows
about form, Elo, expected goals and fixture congestion, and it knows nothing at
all about what happened this week.

**Branch 2** is an LLM that reads current news about the specific fixture. It
does not judge how much a player matters. It only extracts who is named and
what happened to them, and then a squad importance table built from our own
FBref data decides how much weight that carries.

The two get combined in log space at the end.

### What good looks like

Three class football prediction has a hard ceiling. Bookmakers, who have team
news, lineups and money on the line, land around 53 to 55 percent. If this
notebook reports 52 percent, that is a genuinely good model. If it reports 70
percent, something has leaked and we go find it.

That is why the bookmaker odds are in the dataset. Not as a feature, as a
scoreboard.

In [ ]:
import sys
from pathlib import Path

# Notebooks live one level down, so the project root has to go on the path
# before src imports will resolve.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, log_loss, confusion_matrix,
                             classification_report)

from src.config import INTERIM_DIR, PROCESSED_DIR, MODELS_DIR
from src import features as F

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (10, 4)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch", torch.__version__, "| device: cpu (fine for a model this size)")

## 1. Load

Two tables. Results going back to 2006/07, and expected goals from 2014/15
onwards. The xG table is smaller, and that gap is deliberate rather than a
mistake: Understat simply did not exist before then.

In [ ]:
raw_path = INTERIM_DIR / "matches_football_data.csv"
prebuilt_path = PROCESSED_DIR / "matches_features.csv"
FROM_SCRATCH = raw_path.exists()

if FROM_SCRATCH:
    matches = pd.read_csv(raw_path, parse_dates=["date"])
    xg_path = INTERIM_DIR / "xg_understat.csv"
    xg = pd.read_csv(xg_path, parse_dates=["date"]) if xg_path.exists() else None
elif prebuilt_path.exists():
    print("No collected data found, loading the committed feature table instead.")
    print("Run the scripts in scripts/ if you want to rebuild it from the sources.\n")
    dataset = pd.read_csv(prebuilt_path, parse_dates=["date"], low_memory=False)
    matches, xg = dataset, None
else:
    raise FileNotFoundError("Run scripts/01_fetch_results.py first")

print(f"matches: {len(matches):,} rows, {matches['season'].nunique()} seasons")
print(f"         {matches['date'].min().date()} to {matches['date'].max().date()}")
if xg is not None:
    print(f"xg:      {len(xg):,} rows from {xg['date'].min().date()}")

matches.head(3)

In [ ]:
# Baseline rates. Everything the model does has to be judged against these.
split = matches["result"].value_counts(normalize=True)
print("Outcome distribution across 20 seasons")
print(f"  home win  {split.get('H', 0):.1%}")
print(f"  draw      {split.get('D', 0):.1%}")
print(f"  away win  {split.get('A', 0):.1%}")
print()
print(f"Always predicting home would score {split.get('H', 0):.1%}.")
print("That is the number to beat. It is higher than people expect.")

## 2. Features

All of this lives in `src/features.py` so the dashboard can import the exact
same code later. Two models that disagree about what a feature means is a
debugging nightmare nobody needs.

The one rule the whole module is built around: a feature for a match may only
use information that existed before kickoff. Shots and corners from the match
itself are results, not inputs.

### Elo

Standard chess Elo adapted for football. Home advantage is worth 65 rating
points, roughly a third of a goal. The margin of victory scales the update, so
a 4-0 moves the ratings more than a 1-0. Between seasons every club gets pulled
25 percent toward the mean, because squads turn over.

Clubs with no history start at 1400 rather than the 1500 average. Coventry are
in this season's fixture list and have not been in the Premier League since
2000/01, so without that rule they would be seeded as a mid-table side.

In [ ]:
if FROM_SCRATCH:
    dataset = F.build_dataset(matches, xg=xg)
print(f"{len(dataset):,} matches, {dataset.shape[1]} columns")

feat_cols = F.feature_columns(dataset)
F.assert_no_leakage(feat_cols)   # raises if a post-match column got in
print(f"{len(feat_cols)} features passed the leakage check")

In [ ]:
# Does Elo actually know anything? If the ratings are meaningful, higher
# elo_diff should mean more home wins. This is the single best sanity check
# available before any training happens.
bins = pd.cut(dataset["elo_diff"], bins=[-600, -200, -100, 0, 100, 200, 600])
check = dataset.groupby(bins, observed=True)["result"].value_counts(normalize=True).unstack()

ax = check[["H", "D", "A"]].plot(kind="bar", stacked=True,
                                 color=["#2e7d32", "#9e9e9e", "#c62828"])
ax.set_title("Outcome by Elo difference (home minus away)")
ax.set_xlabel("Elo difference")
ax.set_ylabel("share of matches")
ax.legend(["home win", "draw", "away win"], loc="upper left",
          bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

check.round(3)

In [ ]:
# Current Elo table. Compare it against your own sense of the league.
# If a mid-table side is top, something in the update loop is wrong.
F.final_elo_table(dataset).head(12)

## 3. Cleaning up the early rows

A team's first few matches in the dataset have no form history behind them, so
the rolling columns are NaN. That is honest rather than broken, but the model
cannot train on it.

Two decisions, both deliberate:

We drop the opening season entirely, since every club in it starts from
nothing. We impute the rest with medians computed **on the training set only**.
Fitting an imputer on all the data would leak future information backwards,
which is a subtle version of exactly the mistake this notebook is built to
avoid.

In [ ]:
first_season = dataset["season_start"].min()
model_df = dataset[dataset["season_start"] > first_season].copy()
print(f"Dropped {first_season}/{(first_season+1)%100:02d}: "
      f"{len(dataset) - len(model_df):,} rows")

# Split by season, never randomly.
#
# Two validation seasons rather than one. A single season is 380 matches,
# which is too noisy to pick an early stopping epoch from: the epoch it
# lands on is partly luck. Doubling it costs 380 training rows and buys a
# much steadier signal.
TEST_SEASON = int(model_df["season_start"].max())
VAL_SEASONS = [TEST_SEASON - 2, TEST_SEASON - 1]

train = model_df[model_df["season_start"] < min(VAL_SEASONS)]
val   = model_df[model_df["season_start"].isin(VAL_SEASONS)]
test  = model_df[model_df["season_start"] == TEST_SEASON]

print(f"train  {len(train):,}  (up to {min(VAL_SEASONS)-1}/{min(VAL_SEASONS)%100:02d})")
print(f"val    {len(val):,}  ({VAL_SEASONS[0]}/{(VAL_SEASONS[0]+1)%100:02d} "
      f"and {VAL_SEASONS[1]}/{(VAL_SEASONS[1]+1)%100:02d})")
print(f"test   {len(test):,}  ({TEST_SEASON}/{(TEST_SEASON+1)%100:02d})")

In [ ]:
medians = train[feat_cols].median()

def prep(df):
    X = df[feat_cols].fillna(medians).to_numpy(dtype=np.float32)
    y = df["target"].to_numpy(dtype=np.int64)
    return X, y

X_train, y_train = prep(train)
X_val,   y_val   = prep(val)
X_test,  y_test  = prep(test)

scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = (scaler.transform(a) for a in (X_train, X_val, X_test))

print("shapes:", X_train_s.shape, X_val_s.shape, X_test_s.shape)
print("no NaNs left:", not np.isnan(X_train_s).any())

## 4. The bookmaker benchmark

Before training anything, work out what Bet365 achieved on the test season.
Their odds carry an overround (the margin), so the three implied probabilities
sum to more than one. Normalising removes it.

This number is the honest yardstick. Everything below gets compared to it.

In [ ]:
book_test = F.implied_probabilities(test)
mask = book_test.notna().all(axis=1)

book_probs = book_test[mask].to_numpy()
book_pred = book_probs.argmax(axis=1)
book_true = y_test[mask.to_numpy()]

BOOK = {
    "accuracy": accuracy_score(book_true, book_pred),
    "log_loss": log_loss(book_true, book_probs, labels=[0, 1, 2]),
}
print(f"Bookmaker on {TEST_SEASON}/{(TEST_SEASON+1)%100:02d} "
      f"({mask.sum()} matches with odds)")
print(f"  accuracy  {BOOK['accuracy']:.1%}")
print(f"  log loss  {BOOK['log_loss']:.4f}")
print()
print("Note how rarely they pick the draw:")
print(pd.Series(book_pred).map({0: 'home', 1: 'draw', 2: 'away'}).value_counts())

## 5. Baselines

Two of them, and they are not a formality. On tabular data of this size,
gradient boosting frequently beats a neural network. Reporting that honestly is
better work than pretending the network won.

In [ ]:
logreg = LogisticRegression(max_iter=2000, C=0.1)
logreg.fit(X_train_s, y_train)

p = logreg.predict_proba(X_test_s)
LOGREG = {"accuracy": accuracy_score(y_test, p.argmax(1)),
          "log_loss": log_loss(y_test, p, labels=[0, 1, 2])}
print(f"Logistic regression   acc {LOGREG['accuracy']:.1%}   "
      f"log loss {LOGREG['log_loss']:.4f}")

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softprob", num_class=3,
    eval_metric="mlogloss", early_stopping_rounds=40,
    random_state=SEED,
)
xgb.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)

p = xgb.predict_proba(X_test_s)
XGB = {"accuracy": accuracy_score(y_test, p.argmax(1)),
       "log_loss": log_loss(y_test, p, labels=[0, 1, 2])}
print(f"XGBoost               acc {XGB['accuracy']:.1%}   "
      f"log loss {XGB['log_loss']:.4f}")
print(f"stopped at {xgb.best_iteration} trees")

In [ ]:
# Which features are carrying the model? Useful for the report, and it
# catches nonsense: if matchweek is at the top, something is wrong.
imp = pd.Series(xgb.feature_importances_, index=feat_cols).nlargest(15)
ax = imp.sort_values().plot(kind="barh", color="#1565c0")
ax.set_title("XGBoost feature importance, top 15")
plt.tight_layout(); plt.show()

## 6. The neural network

Small on purpose. Around 6,000 training rows does not support anything deep,
and a bigger network just memorises which seasons Leicester were good.

Two hidden layers, batch norm, dropout at 0.3, softmax over three classes.

**Class weights matter here.** Draws are about a quarter of matches and are
genuinely hard to predict, so an unweighted network learns the easy trick of
never predicting one. Weighting the loss by inverse class frequency forces it
to at least try. Watch the confusion matrix rather than accuracy to see whether
it worked.

In [ ]:
from src.predict import MatchNet

# Class weights, softened.
#
# Straight inverse frequency is too aggressive. It makes the network
# predict draws it has no business predicting, and it pushes probability
# mass away from home and away, which shows up later as a badly
# calibrated model.
#
# Raising the weights to a power below 1 keeps a nudge toward draws
# without letting them take over. 0.5 is a good starting point. If draws
# disappear from the confusion matrix entirely, try 0.7. If the model
# predicts far more draws than it gets right, drop toward 0.3.
DRAW_WEIGHT_POWER = 0.5

counts = np.bincount(y_train, minlength=3)
raw = len(y_train) / (3 * counts)
softened = raw ** DRAW_WEIGHT_POWER
class_weights = torch.tensor(softened / softened.mean(), dtype=torch.float32)

print("raw inverse frequency:", raw.round(3))
print(f"softened (power {DRAW_WEIGHT_POWER}):", class_weights.numpy().round(3))
print()
print("Draws are about a quarter of matches, so they still get the most")
print("weight. They just no longer dominate the loss.")

In [ ]:
def train_network(X_tr, y_tr, X_va, y_va, weights,
                  epochs=300, lr=1e-3, patience=30, verbose=True):
    model = MatchNet(X_tr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    Xtr = torch.tensor(X_tr, dtype=torch.float32)
    ytr = torch.tensor(y_tr)
    Xva = torch.tensor(X_va, dtype=torch.float32)
    yva = torch.tensor(y_va)

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

    best, best_state, wait = np.inf, None, 0
    history = []

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            tr_loss = loss_fn(model(Xtr), ytr).item()
            va_loss = loss_fn(model(Xva), yva).item()
        history.append((tr_loss, va_loss))

        if va_loss < best - 1e-4:
            best, wait = va_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"early stop at epoch {epoch}, best val loss {best:.4f}")
                break

        if verbose and epoch % 25 == 0:
            print(f"  epoch {epoch:3d}   train {tr_loss:.4f}   val {va_loss:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    return model, np.array(history)


model, history = train_network(X_train_s, y_train, X_val_s, y_val, class_weights)

In [ ]:
plt.plot(history[:, 0], label="train")
plt.plot(history[:, 1], label="validation")
plt.xlabel("epoch"); plt.ylabel("weighted cross entropy")
plt.title("Training curve")
plt.legend(); plt.tight_layout(); plt.show()

print("If validation flattens early while train keeps falling, that is")
print("overfitting and early stopping did its job.")

### Temperature calibration

The network almost certainly ranks matches correctly and then squashes its
probabilities toward the middle. That is what class weighting does, and it is
easy to miss because accuracy does not change when you fix it.

One parameter fixes it. Divide the logits by a temperature before the softmax:
below 1 sharpens an underconfident model, above 1 softens an overconfident one.

Fit it on the validation seasons, apply it to test. Fitting it on test is the
same mistake as tuning any other hyperparameter there, just less obvious
because it is a single number.

This matters more here than in most projects. The news branch shifts these
probabilities, and shifting a squashed number gives you a wrong number moved
slightly.

In [ ]:
from src.predict import fit_temperature

with torch.no_grad():
    val_logits = model(torch.tensor(X_val_s, dtype=torch.float32)).numpy()
    test_logits = model(torch.tensor(X_test_s, dtype=torch.float32)).numpy()

TEMPERATURE = fit_temperature(val_logits, y_val)

def apply_temp(logits, t=None):
    t = TEMPERATURE if t is None else t
    return torch.softmax(torch.tensor(logits, dtype=torch.float32) / t, dim=1).numpy()

nn_probs_raw = apply_temp(test_logits, 1.0)
nn_probs     = apply_temp(test_logits)

print(f"temperature {TEMPERATURE:.3f}")
print("  below 1 means the model was underconfident and got sharpened")
print("  above 1 means it was overconfident and got softened\n")
print(f"log loss before  {log_loss(y_test, nn_probs_raw, labels=[0,1,2]):.4f}")
print(f"log loss after   {log_loss(y_test, nn_probs, labels=[0,1,2]):.4f}")
print(f"accuracy         {accuracy_score(y_test, nn_probs.argmax(1)):.1%} "
      f"(unchanged, temperature cannot reorder predictions)")

## 7. Evaluation

Accuracy alone is a bad measure here, because the merge step needs
probabilities that mean something. So three numbers:

**Accuracy** is how often the top pick is right.

**Log loss** punishes confident mistakes. It is what actually matters for the
merge, since a badly calibrated network makes the news adjustment meaningless.

**The confusion matrix** shows whether draws are ever predicted at all.

In [ ]:
def evaluate(probs, y_true, name):
    pred = probs.argmax(1)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "log_loss": log_loss(y_true, probs, labels=[0, 1, 2]),
        "draws_predicted": int((pred == 1).sum()),
    }

# nn_probs is already temperature corrected from the cell above.
rows = [
    evaluate(np.repeat([[0.45, 0.25, 0.30]], len(y_test), axis=0), y_test,
             "always home"),
    evaluate(logreg.predict_proba(X_test_s), y_test, "logistic regression"),
    evaluate(xgb.predict_proba(X_test_s), y_test, "xgboost"),
    evaluate(nn_probs_raw, y_test, "neural net (raw)"),
    evaluate(nn_probs, y_test, "neural net (calibrated)"),
    {"model": "bookmaker", **BOOK,
     "draws_predicted": int((book_pred == 1).sum())},
]
results = pd.DataFrame(rows)
results["accuracy"] = (results["accuracy"] * 100).round(1)
results["log_loss"] = results["log_loss"].round(4)

print("The bookmaker row is the yardstick. Some seasons are more")
print("predictable than others, so read your gap to them rather than")
print("the raw number.\n")
results

In [ ]:
cm = confusion_matrix(y_test, nn_probs.argmax(1), labels=[0, 1, 2])
labels = ["home", "draw", "away"]

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), labels); ax.set_yticks(range(3), labels)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Neural network, test season")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()

print(classification_report(y_test, nn_probs.argmax(1),
                            target_names=labels, zero_division=0))

### Calibration

When the model says 60 percent, does it happen 60 percent of the time?

This matters more than usual here. The news branch shifts these probabilities,
and shifting a number that does not mean anything produces a different number
that also does not mean anything.

In [ ]:
def calibration(probs, y_true, cls=0, bins=8):
    p = probs[:, cls]
    hit = (y_true == cls).astype(int)
    edges = np.linspace(0, 1, bins + 1)
    idx = np.digitize(p, edges) - 1
    xs, ys, ns = [], [], []
    for b in range(bins):
        m = idx == b
        if m.sum() >= 15:
            xs.append(p[m].mean()); ys.append(hit[m].mean()); ns.append(int(m.sum()))
    return xs, ys, ns

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, probs, title in [(axes[0], nn_probs_raw, "before calibration"),
                         (axes[1], nn_probs, "after calibration")]:
    ax.plot([0, 1], [0, 1], "k--", label="perfect")
    for cls, name, colour in [(0, "home", "#2e7d32"), (2, "away", "#c62828")]:
        xs, ys, _ = calibration(probs, y_test, cls)
        ax.plot(xs, ys, "o-", color=colour, label=name)
    ax.set_title(title); ax.set_xlabel("predicted probability")
axes[0].set_ylabel("observed frequency"); axes[0].legend()
plt.tight_layout(); plt.show()

print("Points above the line mean underconfident, below means overconfident.")
print("The right panel should hug the diagonal more closely than the left.\n")
for cls, name in [(0, "home"), (2, "away")]:
    xs, ys, ns = calibration(nn_probs, y_test, cls)
    print(f"{name}:")
    for x, y, n in zip(xs, ys, ns):
        print(f"  predicted {x:.2f} -> observed {y:.2f}  (n={n})")

## 8. Save the model

Weights, scaler statistics and feature order all travel together. A saved
network without the exact feature order that produced it is unusable, and the
failure is silent: it will happily predict nonsense from mismatched columns.

In [ ]:
from src.predict import save_bundle

metrics = {
    "test_season": f"{TEST_SEASON}/{(TEST_SEASON+1)%100:02d}",
    "accuracy": float(accuracy_score(y_test, nn_probs.argmax(1))),
    "log_loss": float(log_loss(y_test, nn_probs, labels=[0, 1, 2])),
    "log_loss_uncalibrated": float(log_loss(y_test, nn_probs_raw, labels=[0, 1, 2])),
    "bookmaker_accuracy": float(BOOK["accuracy"]),
    "bookmaker_log_loss": float(BOOK["log_loss"]),
    "draw_weight_power": DRAW_WEIGHT_POWER,
    "n_train": int(len(y_train)),
}

# The temperature goes into the bundle. A model saved without it produces
# probabilities that rank correctly and mean nothing.
save_bundle(model, scaler, feat_cols, metrics, temperature=TEMPERATURE)
dataset.to_csv(PROCESSED_DIR / "matches_features.csv", index=False)

print("saved to models/ and data/processed/\n")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## 9. Squad importance

This is the piece that makes the news branch work.

Importance is 60 percent minutes played and 40 percent goal involvements, both
measured against the top player in that same squad and season. A first choice
striker lands near 0.9. A third choice full back who made four substitute
appearances lands near 0.1.

The LLM never sees this. It extracts names, and this table decides what they
are worth.

In [ ]:
from src import importance as IMP

# If this raises a KeyError, run IMP.inspect_columns() and adjust the
# patterns in src/importance.py to match your FBref column names.
importance = IMP.build_importance()
print(f"{len(importance):,} player seasons")

latest = int(importance["season_start"].max())
print(f"\nMost important players, {latest}/{(latest+1)%100:02d}:")
importance[importance["season_start"] == latest].nlargest(10, "importance")[
    ["player", "team", "position", "minutes", "goals", "assists", "importance"]
]

In [ ]:
# Sanity check on a single squad. The spread across the list is the whole
# point: if everyone scored 0.8 the branch could not tell anyone apart.
team = importance[importance["season_start"] == latest]["team"].iloc[0]
print(f"{team}, {latest}/{(latest+1)%100:02d}\n")
IMP.squad(importance, team, latest, top=20)

## 10. The news branch

Three stages, kept separate on purpose.

**Search** finds recent articles through DuckDuckGo. Free, no key, and it fails
gracefully to an empty list.

**Extract** hands those snippets to the LLM with one job: return who is named,
which side they play for, and what happened to them. No opinions and no
predictions. This is the thing language models are actually reliable at.

**Score** looks each name up in the importance table above and converts it to a
number per team.

Asking the LLM to judge importance directly is the obvious design and the wrong
one. It has no idea who is currently first choice at Brentford, and it will
invent an answer rather than admit that.

In [ ]:
from src import news as N

HOME, AWAY = "Arsenal", "Chelsea"

articles = N.search_news(HOME, AWAY)
print(f"{len(articles)} articles found\n")
for a in articles[:5]:
    print(f"  {a['title'][:90]}")

if not articles:
    print("\nNothing found. Check your connection, or try a fixture that is")
    print("actually coming up soon.")

In [ ]:
# This is the first call that uses your API key.
items = N.extract_facts(articles, HOME, AWAY)

print("Extracted facts:")
for i in items:
    print(f"  {i['player']:26} {i['side']:5} {i['status']}")
if not items:
    print("  none, which is a valid answer if there is no team news")

In [ ]:
home_score, away_score, facts = N.score_facts(items, importance, HOME, AWAY)

print(f"{HOME:20} news score {home_score:+.3f}")
print(f"{AWAY:20} news score {away_score:+.3f}\n")
for f in sorted(facts, key=lambda x: -abs(x.impact)):
    print(f"  {f.player:26} {f.status:11} "
          f"importance {f.importance:.2f}   impact {f.impact:+.2f}")

print("\nLook at the importance column. That separation between a key player")
print("and a squad player is the entire reason this branch exists.")

## 11. Merging the branches

In log space, not by averaging probabilities.

```
delta  = news_home - news_away
adjust = [alpha * delta, 0, -alpha * delta]
final  = softmax(log(p_network) + adjust)
```

With alpha at 0.4, a key striker ruled out moves the home probability by about
13 points, a fringe player moves it by 1.5, and the worst possible news cannot
flip a heavy favourite.

That bound is deliberate. We cannot validate this branch historically, because
there is no archive of labelled pre-match news to test it against. An
unvalidated branch gets to nudge, not decide. Being explicit about that in the
report is a strength rather than an admission.

In [ ]:
from src.predict import merge_branches

demo = nn_probs[0]
print(f"network says: home {demo[0]:.1%}  draw {demo[1]:.1%}  away {demo[2]:.1%}\n")

for label, nh, na in [
    ("no news",                  0.0,  0.0),
    ("home key striker out",    -0.9,  0.0),
    ("home fringe player out",  -0.1,  0.0),
    ("away key striker out",     0.0, -0.9),
    ("both sides hit equally",  -0.8, -0.8),
]:
    m = merge_branches(demo, nh, na)
    print(f"{label:26} home {m[0]:.1%}  draw {m[1]:.1%}  away {m[2]:.1%}"
          f"   (home {m[0]-demo[0]:+.1%})")

## 12. End to end

One function, both branches, the same one the dashboard will call.

It degrades safely: no API key, no internet or no articles gives you the
network prediction and a message saying why. A broken news branch must never
take the whole prediction down.

In [ ]:
from src.predict import predict_match
from src.news import NewsResult

# Reuse the analysis from the cells above rather than searching again.
# Firing several DuckDuckGo searches back to back gets you rate limited,
# and a rate limited search returns nothing, which silently collapses the
# news score to zero. That is not an error you would notice: you just get
# the network prediction and a neutral adjustment.
cached_news = NewsResult(home_score=home_score, away_score=away_score,
                         facts=facts, articles=articles)

result = predict_match(HOME, AWAY, dataset, importance_table=importance,
                       news_result=cached_news)
print(result)
print()
print("News reasoning:")
print(result.news_explanation)
print()
if abs(result.news_home - result.news_away) < 1e-9:
    print("Both news scores are zero. Either there genuinely was no team")
    print("news, or the search came back empty. Check the articles list.")

In [ ]:
# Run it across the next few real fixtures.
fx_path = PROCESSED_DIR / "fixtures_upcoming.csv"

if fx_path.exists():
    fixtures = pd.read_csv(fx_path, parse_dates=["date"]).head(5)
    for _, fx in fixtures.iterrows():
        try:
            r = predict_match(fx["home_team"], fx["away_team"], dataset,
                              importance_table=importance, use_news=False)
            print(f"{fx['date'].date()}  {r}")
            print()
        except ValueError as exc:
            print(f"{fx['home_team']} vs {fx['away_team']}: {exc}\n")
else:
    print("Run scripts/05_fetch_fixtures.py first.")

## Where this leaves us

The trained network, its scaler and the feature order are in `models/`. The
feature table and the importance table are in `data/processed/`. Everything the
dashboard needs is importable from `src/`.

### Honest limitations, worth writing up

**The form features are as fresh as the data.** If collection stopped in May
and you predict in September, "recent form" means last season's final matches.
Bump `LAST_SEASON` in `src/config.py` and re-run collection with `--force`.

**The news branch has never been validated.** No labelled archive exists to
test it against. That is why alpha is small and every component is displayed
separately rather than hidden inside one number.

**Newly promoted clubs are guesses.** Coventry get a 1400 Elo prior and nothing
else. Their first few predictions will be poor and there is no honest way
around that with league-only data.

**Beating the bookmaker is not the target.** They see confirmed lineups an hour
before kickoff. Landing within a couple of points of them is the real result.

### Next

Notebook 2, market value, which also improves this one: market value is a
better importance signal than minutes and goals, and it drops straight into the
news branch.